# GFP 도킹과 접힘 진단 · v0.1.10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/notebooks/diagnostics/GFP_Diagnostics_T4.ipynb?v=0.1.10)

T4 GPU 런타임에서 Minimizer 도킹과 자동 후보 선택을 실행합니다. Enhancer 접힘과 복합체 배치도 비교할 수 있습니다.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")
sys.path.insert(0, str(checkout / "gfp_pipeline"))
from notebook_workflow import build_jobs, run_jobs, display_results, package_results


## 진단 실험 선택


In [ ]:
DIAGNOSTIC_EXPERIMENT = "minimizer_docking" #@param ["minimizer_docking", "enhancer_split", "all"]
RUN_MODE = "diagnostics"
jobs = build_jobs(checkout, RUN_MODE, diagnostic=DIAGNOSTIC_EXPERIMENT)
completed_runs = []
for job in jobs:
    print(job["label"])


## 실험 실행

상세 출력은 실험별 로그에 저장하고 실패 시 마지막 40줄을 표시합니다. 같은 세션의 설치 환경과 모델 가중치를 재사용합니다. NanoBodyBuilder2·MEGADOCK·RF3는 별도 환경에서 실행됩니다.


In [ ]:
run_jobs(checkout, revision, jobs, completed_runs)


## 결과 비교


In [ ]:
display_results(checkout, RUN_MODE, completed_runs)


## 결과 다운로드

구조·지표·실제 패키지 목록·로그를 ZIP으로 내려받습니다. 중간 실험이 실패해도 완료된 실험이 있으면 이 셀에서 보존할 수 있습니다.


In [ ]:
from google.colab import files
result_zip = package_results(revision, RUN_MODE, completed_runs)
print(result_zip)
files.download(str(result_zip))
